Imports

In [4]:
!pip install wordcloud
!pip install nltk


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------- ---------------------- 0.8/1.8 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 4.3 MB/s  0:00:00



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [6]:
df = pd.read_csv(r'C:\Users\ASUS\OneDrive\Pictures\Desktop\coding world\MLOps_Vikash_dash\MLOPS-Complete-ML-Pipeline\experiments\spam.csv')

In [7]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [8]:
df = df.drop(columns = ['Unnamed: 2','Unnamed: 3','Unnamed: 4'])

In [9]:
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [10]:
df.rename(columns = {'v1':'target','v2':'text'},inplace = True)
df.head()

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Data Preprocessing

In [11]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
df['target'] = encoder.fit_transform(df['target'])
df.head()

,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [13]:
df.duplicated().sum()

np.int64(403)

In [14]:
df = df.drop_duplicates(keep = 'first')

In [16]:
len(df)

5169

Feature Engineering

In [18]:
from nltk.stem.porter import PorterStemmer

import string
ps = PorterStemmer()

In [19]:
def transform_text(text):
    text = text.lower()
    text = nltk.word_tokenize(text)
    
    # removing special character
    y = []
    for i in text:
        if i.isalnum():
            y.append(i)
    
    # removing stop words and punctuations
    text = y[:]
    y.clear()
    
    # Loop through the tokens and remove stopwords and punctuations
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)
    
    # stemming using Porterstemmer
    text = y[:]
    y.clear()
    for i in text:
        y.append(ps.stem(i))
        
    # Join the processed back into a single string
    return " ".join(y)

In [20]:
df['transformed_text'] = df['text'].apply(transform_text)
df.head()

,target,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


In [22]:
from sklearn.feature_extraction.text import CountVectorizer , TfidfVectorizer

In [23]:
tfidf = TfidfVectorizer(max_features = 500)

In [24]:
x = tfidf.fit_transform(df['transformed_text']).toarray()
y = df['target'].values

Train test split

In [25]:
from sklearn.model_selection import train_test_split

In [26]:
x_train , x_test  , y_train , y_test = train_test_split(x , y, test_size = 0.2 , stratify = y , random_state = 42)

Model training

In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier , AdaBoostClassifier , BaggingClassifier , ExtraTreesClassifier , GradientBoostingClassifier
from xgboost import XGBClassifier

In [32]:
svc = SVC(kernel = 'sigmoid' , gamma = 1.0)
knc = KNeighborsClassifier()
mnb = MultinomialNB()
dtc = DecisionTreeClassifier(max_depth = 5)
lrc = LogisticRegression(solver  ='liblinear' , penalty = 'l1')
rfc = RandomForestClassifier(n_estimators = 50 , random_state = 2)
abc = AdaBoostClassifier(n_estimators = 50 , random_state = 2)
bc = BaggingClassifier(n_estimators = 50 , random_state = 2)
etc = ExtraTreesClassifier(n_estimators = 50  , random_state = 2)
gbc = GradientBoostingClassifier(n_estimators = 50  , random_state = 2)
xgb = XGBClassifier(n_estimators = 50 , random_state = 2)

In [34]:
clfs = {
    'svc':svc,
    'KNN':knc,
    'NB': mnb,
    'DT':dtc,
    'LR':lrc,
    'RF':rfc,
    'AB':abc,
    'bgc':bc,
    'ETC':etc,
    'GBC':gbc,
    'xgb':xgb
}

In [35]:
from sklearn.metrics import accuracy_score , precision_score
def Model_train(clfs , x_train , y_train , x_test , y_test):
    clfs.fit(x_train , y_train)
    y_pred = clfs.predict(x_test)
    accuracy = accuracy_score(y_test , y_pred)
    precision = precision_score(y_test , y_pred)
    return accuracy , precision

In [36]:
accuracy_scores = []
precision_scores = []

for name , clfs in clfs.items():
    current_accuracy , current_precision = Model_train(clfs , x_train , y_train , x_test , y_test)
    print()
    print("For: ",name)
    print("Accuracy :",current_accuracy)
    print("Precision: ",current_precision)
    
    accuracy_scores.append(current_accuracy)
    precision_scores.append(current_precision)
    


For:  svc
Accuracy : 0.9748549323017408
Precision:  0.9565217391304348

For:  KNN
Accuracy : 0.9274661508704062
Precision:  1.0

For:  NB
Accuracy : 0.9709864603481625
Precision:  0.963302752293578

For:  DT
Accuracy : 0.9390715667311412
Precision:  0.8617021276595744

For:  LR
Accuracy : 0.9690522243713733
Precision:  0.9626168224299065


c:\Users\ASUS\OneDrive\Pictures\Desktop\coding world\Deep Learning in Tensorflow\DL_campusx\dl_env\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\ASUS\OneDrive\Pictures\Desktop\coding world\Deep Learning in Tensorflow\DL_campusx\dl_env\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(



For:  RF
Accuracy : 0.9777562862669246
Precision:  0.9655172413793104

For:  AB
Accuracy : 0.9245647969052224
Precision:  0.7789473684210526

For:  bgc
Accuracy : 0.9632495164410058
Precision:  0.8661417322834646

For:  ETC
Accuracy : 0.9748549323017408
Precision:  0.9338842975206612

For:  GBC
Accuracy : 0.9584139264990329
Precision:  0.9680851063829787

For:  xgb
Accuracy : 0.97678916827853
Precision:  0.9734513274336283
